# 01 - Data and Problem Overview

This notebook prepares the cleaned dataset used by the later modeling notebooks and explains the structure of the drive-failure prediction problem.

It covers:

1. loading the public Backblaze Q4 2023 daily drive-stat files;
2. selecting five candidate raw SMART attributes;
3. examining class imbalance, repeated drive observations, missingness, and value distributions;
4. removing `smart_187_raw` and `smart_188_raw`;
5. filling the remaining missing SMART values with zero;
6. validating and saving `df_cleaned.parquet`.

No model is selected in this notebook. Model comparison and leakage-safe cross-validation begin in Notebook 02.

## Project and data scope

The project uses public Backblaze drive telemetry accessed through a Kaggle re-upload. Each row represents one physical drive observed on one date—a **drive-day** observation.

The same `serial_number` can therefore appear on many dates. This repeated-observation structure creates a validation risk: a random row split could place observations from the same physical drive in both training and validation data. Notebook 02 addresses that issue with grouped cross-validation.

The source data primarily represents Backblaze's storage-drive fleet and should not be interpreted as proprietary Silicon Motion data or direct validation on Silicon Motion SSD hardware.

In [ ]:
from pathlib import Path
import glob

import matplotlib.pyplot as plt
import pandas as pd

# Kaggle input location used during the original project.
DATA_GLOB = (
    "/kaggle/input/datasets/priyamsaha17/"
    "backblaze-hard-drive-failure-dataset-2023/"
    "data_Q4_2023/data_Q4_2023/*.csv"
)

OUTPUT_PATH = Path("/kaggle/working/df_cleaned.parquet")
RANDOM_STATE = 42
PLOT_SAMPLE_SIZE = 500_000

CANDIDATE_SMART_FEATURES = [
    "smart_5_raw",
    "smart_187_raw",
    "smart_188_raw",
    "smart_197_raw",
    "smart_198_raw",
]

FINAL_SMART_FEATURES = [
    "smart_5_raw",
    "smart_197_raw",
    "smart_198_raw",
]

DROPPED_SMART_FEATURES = [
    "smart_187_raw",
    "smart_188_raw",
]

RAW_COLUMNS = [
    "date",
    "serial_number",
    "model",
    "capacity_bytes",
    "failure",
    *CANDIDATE_SMART_FEATURES,
]

DTYPE_MAP = {
    "serial_number": "string",
    "model": "category",
    "capacity_bytes": "float64",
    "failure": "int8",
    "smart_5_raw": "float32",
    "smart_187_raw": "float32",
    "smart_188_raw": "float64",
    "smart_197_raw": "float32",
    "smart_198_raw": "float32",
}

## 1. Locate and inspect the source files

The project uses the Q4 2023 daily CSV files. The checks below fail clearly if the expected Kaggle dataset is not mounted.

In [ ]:
q4_files = sorted(glob.glob(DATA_GLOB))

if not q4_files:
    raise FileNotFoundError(
        "No Q4 2023 CSV files were found. "
        "Check DATA_GLOB and confirm that the Kaggle dataset is attached."
    )

print(f"Daily CSV files found: {len(q4_files):,}")
print(f"First file: {Path(q4_files[0]).name}")
print(f"Last file:  {Path(q4_files[-1]).name}")

In [ ]:
sample_df = pd.read_csv(
    q4_files[0],
    usecols=RAW_COLUMNS,
    dtype=DTYPE_MAP,
    parse_dates=["date"],
)

print(f"Sample file shape: {sample_df.shape}")
display(sample_df.head())

schema_preview = pd.DataFrame(
    {
        "dtype": sample_df.dtypes.astype(str),
        "missing_rows": sample_df.isna().sum(),
        "missing_percent": sample_df.isna().mean().mul(100),
    }
)
display(schema_preview)

## 2. Candidate SMART attributes

The initial extraction retained five raw SMART attributes commonly associated with drive-health degradation:

| Attribute | General interpretation |
|---|---|
| `smart_5_raw` | Reallocated sector count |
| `smart_187_raw` | Reported uncorrectable errors |
| `smart_188_raw` | Command timeout count |
| `smart_197_raw` | Current pending sector count |
| `smart_198_raw` | Offline uncorrectable sector count |

All five were retained initially so their coverage, missingness, and value distributions could be evaluated before the final cleaned feature set was chosen.

## 3. Load and merge the daily files

Only the required identifier, target, metadata, and candidate SMART columns are loaded. Explicit data types reduce memory usage while the daily files are combined.

In [ ]:
frames = []

for index, file_path in enumerate(q4_files, start=1):
    daily_df = pd.read_csv(
        file_path,
        usecols=RAW_COLUMNS,
        dtype=DTYPE_MAP,
        parse_dates=["date"],
    )
    frames.append(daily_df)

    if index % 20 == 0 or index == len(q4_files):
        print(f"Loaded {index:,}/{len(q4_files):,} files")

df = pd.concat(frames, ignore_index=True)
del frames

print(f"Combined dataframe shape: {df.shape}")
print(
    f"Approximate memory usage: "
    f"{df.memory_usage(deep=True).sum() / 1e9:.2f} GB"
)
display(df.head())

## 4. Problem structure and class imbalance

The target is binary:

- `failure = 0`: no recorded failure on that drive-day;
- `failure = 1`: recorded failure on that drive-day.

Failure observations are extremely rare. Ordinary accuracy would therefore be misleading because a model could predict nearly every row as healthy and still appear highly accurate.

In [ ]:
total_rows = len(df)
unique_drives = df["serial_number"].nunique()
failure_rows = int(df["failure"].sum())
failed_drives = df.loc[df["failure"].eq(1), "serial_number"].nunique()
failure_rate = df["failure"].mean()
average_rows_per_drive = total_rows / unique_drives

print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Drive-day rows: {total_rows:,}")
print(f"Unique drives: {unique_drives:,}")
print(f"Drive models: {df['model'].nunique():,}")
print(f"Failure rows: {failure_rows:,}")
print(f"Unique drives with a recorded failure: {failed_drives:,}")
print(f"Row-level failure prevalence: {failure_rate:.6%}")
print(f"Average observations per drive: {average_rows_per_drive:.1f}")

### Why repeated rows matter

A drive may contribute many daily rows. These rows are not independent observations of different physical devices.

For this reason, later validation groups observations by `serial_number` so that one physical drive cannot appear in both the training and validation portions of the same fold.

## 5. Missing-value analysis

Missingness is evaluated before any values are filled. This step determines whether each candidate attribute has sufficient and consistent coverage for the baseline dataset.

In [ ]:
missing_summary = pd.DataFrame(
    {
        "missing_rows": df[CANDIDATE_SMART_FEATURES].isna().sum(),
        "missing_percent": (
            df[CANDIDATE_SMART_FEATURES].isna().mean().mul(100)
        ),
        "non_missing_rows": df[CANDIDATE_SMART_FEATURES].notna().sum(),
    }
).sort_values("missing_percent", ascending=False)

display(missing_summary.round({"missing_percent": 3}))

`smart_187_raw` and `smart_188_raw` were missing in more than half of the drive-day observations. By comparison, the missing rates for `smart_5_raw`, `smart_197_raw`, and `smart_198_raw` were much lower.

High missingness does not automatically prove that a field is unusable, but it increases uncertainty and can reflect model-specific telemetry availability. The next check examines whether `smart_187_raw` coverage varies systematically across drive models.

In [ ]:
smart_187_by_model = (
    df.groupby("model", observed=True)
    .agg(
        rows=("serial_number", "size"),
        drives=("serial_number", "nunique"),
        smart_187_missing_rate=(
            "smart_187_raw",
            lambda values: values.isna().mean(),
        ),
    )
    .query("rows >= 1000")
    .sort_values("smart_187_missing_rate", ascending=False)
)

print("Models with the highest smart_187_raw missing rates:")
display(smart_187_by_model.head(15))

print("Models with the lowest smart_187_raw missing rates:")
display(smart_187_by_model.tail(15))

The missingness rate differs substantially across drive models. In plain language, `smart_187_raw` is absent systematically for some hardware groups rather than disappearing randomly across the entire dataset.

For the compact baseline developed in this project, the field was dropped instead of introducing model-specific handling for a feature that was unavailable in more than half of the observations.

## 6. SMART sparsity and upper-tail behavior

Raw SMART count fields are typically sparse and strongly right-skewed: most observations are zero, while a small number contain much larger values.

The summaries below examine:

- the share of recorded values equal to zero;
- selected upper quantiles;
- the largest recorded value.

These are descriptive checks, not tests of predictive performance.

In [ ]:
distribution_summary = pd.DataFrame(
    {
        "recorded_rows": df[CANDIDATE_SMART_FEATURES].notna().sum(),
        "zero_percent_of_recorded": (
            df[CANDIDATE_SMART_FEATURES]
            .eq(0)
            .sum()
            .div(df[CANDIDATE_SMART_FEATURES].notna().sum())
            .mul(100)
        ),
        "p50": df[CANDIDATE_SMART_FEATURES].quantile(0.50),
        "p90": df[CANDIDATE_SMART_FEATURES].quantile(0.90),
        "p99": df[CANDIDATE_SMART_FEATURES].quantile(0.99),
        "p999": df[CANDIDATE_SMART_FEATURES].quantile(0.999),
        "maximum": df[CANDIDATE_SMART_FEATURES].max(),
    }
)

display(distribution_summary)

### Additional concern with `smart_188_raw`

In addition to its high missing rate, `smart_188_raw` showed extreme upper-tail values that were difficult to interpret as ordinary command-timeout counts.

Without reliable model-specific decoding rules, the project did not assume that these values represented directly comparable physical counts. The field was excluded rather than interpreted speculatively.

In [ ]:
plot_sample_size = min(PLOT_SAMPLE_SIZE, len(df))
plot_df = df.sample(plot_sample_size, random_state=RANDOM_STATE)

nonzero_rates = (
    plot_df[CANDIDATE_SMART_FEATURES]
    .fillna(0)
    .ne(0)
    .mean()
    .mul(100)
    .sort_values()
)

ax = nonzero_rates.plot(kind="barh", figsize=(8, 4))
ax.set_title(
    f"Non-zero SMART values in a {plot_sample_size:,}-row sample"
)
ax.set_xlabel("Drive-day rows with a non-zero value (%)")
ax.set_ylabel("SMART attribute")
plt.tight_layout()
plt.show()

del plot_df

## 7. Cleaning decisions

The final cleaning decisions were:

| Attribute | Decision | Reason |
|---|---|---|
| `smart_5_raw` | Retain | Low missing rate relative to the dropped fields |
| `smart_187_raw` | Drop | Missing in more than half of observations |
| `smart_188_raw` | Drop | Missing in more than half of observations and difficult-to-interpret extreme values |
| `smart_197_raw` | Retain | Low missing rate relative to the dropped fields |
| `smart_198_raw` | Retain | Low missing rate relative to the dropped fields |

The remaining missing values in `smart_5_raw`, `smart_197_raw`, and `smart_198_raw` are filled with zero.

This is a practical modeling assumption. A missing reading and a genuine recorded zero may not always mean exactly the same thing. The retained fields had comparatively low missingness, and zero represents the absence of a recorded raw event count in the baseline dataset.

In [ ]:
# Drop the two candidate fields excluded during data-quality review.
df_cleaned = df.drop(columns=DROPPED_SMART_FEATURES).copy()

# Fill the remaining baseline SMART fields with zero.
df_cleaned[FINAL_SMART_FEATURES] = (
    df_cleaned[FINAL_SMART_FEATURES].fillna(0)
)

print("Final SMART feature set:")
for feature in FINAL_SMART_FEATURES:
    print(f"  - {feature}")

print("\nRemaining missing values in final SMART features:")
display(df_cleaned[FINAL_SMART_FEATURES].isna().sum())

## 8. Validate the cleaned dataframe

The checks below confirm that:

- the required output columns exist;
- the dropped attributes are no longer present;
- the final SMART fields contain no missing values;
- the target remains binary;
- identifier and date fields are available.

In [ ]:
OUTPUT_COLUMNS = [
    "date",
    "serial_number",
    "model",
    "capacity_bytes",
    "failure",
    *FINAL_SMART_FEATURES,
]

missing_output_columns = set(OUTPUT_COLUMNS) - set(df_cleaned.columns)
if missing_output_columns:
    raise ValueError(
        f"Missing required output columns: {sorted(missing_output_columns)}"
    )

unexpected_dropped_columns = (
    set(DROPPED_SMART_FEATURES).intersection(df_cleaned.columns)
)
if unexpected_dropped_columns:
    raise ValueError(
        "Dropped SMART fields are still present: "
        f"{sorted(unexpected_dropped_columns)}"
    )

assert df_cleaned[FINAL_SMART_FEATURES].isna().sum().sum() == 0
assert df_cleaned["serial_number"].notna().all()
assert df_cleaned["date"].notna().all()
assert df_cleaned["failure"].isin([0, 1]).all()

df_cleaned = df_cleaned[OUTPUT_COLUMNS].copy()

print("Validation passed.")
print(f"Cleaned dataframe shape: {df_cleaned.shape}")
display(df_cleaned.head())

## 9. Save and verify `df_cleaned.parquet`

The file is written only after all cleaning and validation steps are complete. It becomes the input dataset for the grouped model-validation workflow in Notebook 02.

In [ ]:
df_cleaned.to_parquet(OUTPUT_PATH, index=False)

print(f"Saved cleaned dataset to: {OUTPUT_PATH}")
print(f"Saved rows: {len(df_cleaned):,}")
print(f"Saved columns: {len(df_cleaned.columns):,}")

In [ ]:
verification_df = pd.read_parquet(OUTPUT_PATH)

assert list(verification_df.columns) == OUTPUT_COLUMNS
assert verification_df[FINAL_SMART_FEATURES].isna().sum().sum() == 0
assert len(verification_df) == len(df_cleaned)

print("Saved Parquet file successfully verified.")
display(verification_df.head())

## 10. Notebook conclusion

The raw Q4 2023 files were merged into a drive-day dataframe and reduced from five candidate SMART attributes to three final baseline features:

```python
FINAL_SMART_FEATURES = [
    "smart_5_raw",
    "smart_197_raw",
    "smart_198_raw",
]
```

`smart_187_raw` and `smart_188_raw` were excluded because both were missing in more than half of the observations. `smart_188_raw` also contained extreme upper-tail values that could not be interpreted reliably without additional model-specific decoding.

The remaining missing values were filled with zero, the cleaned schema was validated, and the result was saved as:

```text
df_cleaned.parquet
```

The next notebook uses `StratifiedGroupKFold` with `serial_number` as the grouping key to compare candidate models without placing observations from the same physical drive in both training and validation folds.